# 图像增广与迁移学习

本notebook介绍计算机视觉中的两个核心技术:图像增广和迁移学习。

## 学习目标

- 掌握常用的图像增广方法
- 理解图像增广的作用和原理
- 学习迁移学习和模型微调
- 实践:热狗识别任务

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import os

print(f"PyTorch版本: {torch.__version__}")
print(f"Torchvision版本: {torchvision.__version__}")

## 1. 图像增广 (Image Augmentation)

### 1.1 为什么需要图像增广?

**问题**: 深度学习需要大量数据,但获取标注数据成本高

**解决方案**: 图像增广
- **扩充数据集**: 对训练图像随机变换,生成相似但不同的样本
- **提高泛化能力**: 减少模型对特定属性的依赖
- **防止过拟合**: 增加训练样本的多样性

**应用场景**:
- AlexNet的成功离不开图像增广
- 几乎所有CV任务的标配
- 特别适合小数据集

### 1.2 常用增广方法

**几何变换**:
- 翻转 (Flip)
- 裁剪 (Crop)
- 旋转 (Rotate)
- 缩放 (Scale)

**颜色变换**:
- 亮度 (Brightness)
- 对比度 (Contrast)
- 饱和度 (Saturation)
- 色调 (Hue)

**其他**:
- 噪声注入
- 模糊
- 擦除 (Cutout)

In [ ]:
# 加载示例图像
# 如果没有图像,可以从网上下载或使用自己的图像
try:
    img_path = '../img/cat1.jpg'  # 替换为你的图像路径
    img = Image.open(img_path)
    print(f"图像大小: {img.size}")
    
    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title('原始图像')
    plt.show()
except:
    print("图像文件未找到,将创建随机图像演示")
    # 创建随机图像用于演示
    img = Image.fromarray((torch.rand(400, 500, 3).numpy() * 255).astype('uint8'))

# 辅助函数:显示多个增广结果
def show_augmented_images(img, aug_transform, num_samples=8, title="增广示例"):
    """显示多个增广后的图像"""
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.flatten()
    
    for i in range(num_samples):
        augmented = aug_transform(img)
        axes[i].imshow(augmented)
        axes[i].axis('off')
    
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

### 1.3 翻转 (Flip)

In [ ]:
# 左右翻转
print("=== 左右翻转 ===")
flip_lr = transforms.RandomHorizontalFlip(p=1.0)  # p=1.0确保每次都翻转
show_augmented_images(img, flip_lr, num_samples=4, title="左右翻转 (50%概率)")

print("\n通常用于:")
print("  - 几乎所有图像分类任务")
print("  - 对象位置不敏感的任务")
print("  - 不改变语义的情况")

In [ ]:
# 上下翻转
print("=== 上下翻转 ===")
flip_tb = transforms.RandomVerticalFlip(p=1.0)
show_augmented_images(img, flip_tb, num_samples=4, title="上下翻转")

print("\n注意:")
print("  - 使用较少 (不符合自然场景)")
print("  - 适用于医学图像、卫星图像等")
print("  - 可能改变语义(如文字、人脸)")

### 1.4 随机裁剪 (Random Crop)

In [ ]:
# 随机裁剪并调整大小
random_crop = transforms.RandomResizedCrop(
    size=200,           # 输出大小
    scale=(0.1, 1.0),  # 裁剪面积占原图的10%-100%
    ratio=(0.5, 2.0)   # 宽高比范围
)

show_augmented_images(img, random_crop, title="随机裁剪 (不同大小和比例)")

print("\n效果:")
print("  ✅ 对象出现在不同位置 → 位置不变性")
print("  ✅ 不同的缩放比例 → 尺度不变性")
print("  ✅ 降低对目标位置的依赖")
print("\n参数说明:")
print("  - scale: 裁剪面积相对原图的比例")
print("  - ratio: 裁剪区域的宽高比")

### 1.5 颜色变换 (Color Jitter)

In [ ]:
# 亮度变化
brightness_aug = transforms.ColorJitter(brightness=0.5)  # ±50%
show_augmented_images(img, brightness_aug, title="亮度变化 (50%-150%)")

print("亮度范围: [1-0.5, 1+0.5] = [0.5, 1.5]")
print("即原图亮度的50%-150%")

In [ ]:
# 色调变化
hue_aug = transforms.ColorJitter(hue=0.5)
show_augmented_images(img, hue_aug, title="色调变化")

print("\n色调变化:")
print("  - 改变颜色,如红→绿→蓝")
print("  - 降低对颜色的敏感度")

In [ ]:
# 综合颜色变换
color_aug = transforms.ColorJitter(
    brightness=0.5,  # 亮度
    contrast=0.5,    # 对比度
    saturation=0.5,  # 饱和度
    hue=0.5          # 色调
)

show_augmented_images(img, color_aug, title="综合颜色变换")

print("\n四种颜色属性:")
print("  1. 亮度 (Brightness): 整体明暗")
print("  2. 对比度 (Contrast): 最亮和最暗的差异")
print("  3. 饱和度 (Saturation): 色彩鲜艳程度")
print("  4. 色调 (Hue): 颜色类型")

### 1.6 组合多种增广

In [ ]:
# 组合多种增广方法
combined_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),              # 随机水平翻转
    transforms.RandomResizedCrop(200, scale=(0.5, 1.0)),  # 随机裁剪
    transforms.ColorJitter(brightness=0.3, contrast=0.3,  # 颜色变换
                          saturation=0.3, hue=0.2)
])

show_augmented_images(img, combined_aug, title="组合增广 (翻转+裁剪+颜色)")

print("\n实践建议:")
print("  ✅ 根据任务选择合适的增广")
print("  ✅ 不要过度增广(可能改变语义)")
print("  ✅ 训练时增广,测试时不增广")
print("  ✅ 使用Compose组合多种方法")

## 2. 使用图像增广训练

### 2.1 CIFAR-10数据集

**特点**:
- 10类别: 飞机、汽车、鸟、猫、鹿、狗、青蛙、马、船、卡车
- 训练集: 50,000张
- 测试集: 10,000张
- 图像大小: 32×32
- RGB彩色图像

In [ ]:
# 下载CIFAR-10数据集
print("下载CIFAR-10数据集...")
cifar_train = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True)

# 显示前32张图像
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
axes = axes.flatten()

for i in range(32):
    img, label = cifar_train[i]
    axes[i].imshow(img)
    axes[i].axis('off')

plt.suptitle('CIFAR-10 训练集示例', fontsize=14)
plt.tight_layout()
plt.show()

# 类别名称
classes = ('plane', 'car', 'bird', 'cat', 'deer',
          'dog', 'frog', 'horse', 'ship', 'truck')
print(f"\n类别: {classes}")

### 2.2 定义数据增广策略

In [ ]:
# 训练集增广
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # 随机水平翻转
    transforms.RandomCrop(32, padding=4),  # 随机裁剪
    transforms.ToTensor(),  # 转为张量
    transforms.Normalize(  # 标准化
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

# 测试集不增广(只标准化)
test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

print("训练集增广:")
print("  1. 随机水平翻转 (50%概率)")
print("  2. 随机裁剪 (padding=4)")
print("  3. 标准化")
print("\n测试集:")
print("  仅标准化 (确保结果可复现)")

In [ ]:
# 创建数据加载器
batch_size = 128

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, transform=train_transforms, download=True)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, transform=test_transforms, download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                         shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, 
                        shuffle=False, num_workers=2)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")
print(f"批大小: {batch_size}")
print(f"训练批次数: {len(train_loader)}")

### 2.3 简单CNN模型

In [ ]:
# 定义简单的CNN
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = SimpleCNN()
print(model)
print(f"\n参数量: {sum(p.numel() for p in model.parameters()):,}")

### 2.4 训练模型 (演示)

In [ ]:
# 训练函数
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
        
        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)}: '
                  f'Loss={loss.item():.4f}, '
                  f'Acc={100.*correct/total:.2f}%')
    
    return total_loss/len(train_loader), 100.*correct/total

def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    return total_loss/len(test_loader), 100.*correct/total

print("训练函数定义完成!")

In [ ]:
# 演示训练(简短版本)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 2  # 演示用少量epoch
print(f"\n开始训练 ({num_epochs} epochs)...\n")

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_loss, train_acc = train_epoch(model, train_loader, criterion, 
                                       optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    
    print(f"训练: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")
    print(f"测试: Loss={test_loss:.4f}, Acc={test_acc:.2f}%\n")

print("\n注意: 这只是演示,完整训练需要更多epoch")
print("图像增广的效果在更长时间训练后更明显")

## 3. 迁移学习 (Transfer Learning)

### 3.1 什么是迁移学习?

**定义**: 将从源数据集学到的知识迁移到目标数据集

**动机**:
- 目标数据集通常较小
- 从头训练容易过拟合
- 收集和标注数据成本高

**核心思想**:
```
ImageNet (100万+ 图像) → 预训练模型 → 提取通用特征
                              ↓
你的数据集 (几千图像) → 微调模型 → 特定任务
```

### 3.2 微调 (Fine-tuning)

**步骤**:

1. **预训练**: 在大数据集(ImageNet)上训练模型
2. **复制参数**: 创建新模型,复制预训练参数(除输出层)
3. **替换输出层**: 根据目标任务类别数重新初始化
4. **微调训练**: 在目标数据集上训练

**学习率策略**:
- 特征层: 小学习率 (如 0.001)
- 输出层: 大学习率 (如 0.01)
- 原因: 预训练参数已经很好,只需微调

```
┌─────────────────┐
│   ImageNet      │ ← 源数据集 (大)
│   1000类        │
└────────┬────────┘
         │预训练
         ↓
┌─────────────────┐
│  ResNet-18      │ ← 源模型
│  特征层 + FC1000│
└────────┬────────┘
         │复制特征层
         ↓
┌─────────────────┐
│  ResNet-18      │ ← 目标模型
│  特征层 + FC2   │ (替换输出层)
└────────┬────────┘
         │微调
         ↓
┌─────────────────┐
│   热狗数据集    │ ← 目标数据集 (小)
│   2类           │
└─────────────────┘
```

## 4. 实战: 热狗识别

### 4.1 任务描述

**目标**: 识别图像中是否有热狗
- **正类**: 有热狗
- **负类**: 没有热狗

**数据集**:
- 训练集: 1000张 (正负各一半)
- 测试集: 400张
- 较小的数据集 → 适合用迁移学习

**方法**: 使用ImageNet预训练的ResNet-18微调

In [ ]:
# 定义数据增广
# ImageNet的均值和标准差
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# 训练集增广
train_augs = transforms.Compose([
    transforms.RandomResizedCrop(224),      # 随机裁剪到224x224
    transforms.RandomHorizontalFlip(),      # 随机水平翻转
    transforms.ToTensor(),
    normalize
])

# 测试集增广
test_augs = transforms.Compose([
    transforms.Resize(256),                 # 放大到256
    transforms.CenterCrop(224),             # 中心裁剪224x224
    transforms.ToTensor(),
    normalize
])

print("数据增广策略:")
print("\n训练集:")
print("  1. 随机裁剪并调整到224×224")
print("  2. 随机水平翻转")
print("  3. 标准化(ImageNet均值/标准差)")
print("\n测试集:")
print("  1. 调整到256×256")
print("  2. 中心裁剪224×224")
print("  3. 标准化")

### 4.2 加载预训练模型

In [ ]:
# 加载预训练的ResNet-18
print("加载预训练的ResNet-18...")
pretrained_net = torchvision.models.resnet18(pretrained=True)

# 查看原始输出层
print("\n原始输出层 (ImageNet 1000类):")
print(pretrained_net.fc)
print(f"输入特征: {pretrained_net.fc.in_features}")
print(f"输出类别: {pretrained_net.fc.out_features}")

### 4.3 修改输出层

In [ ]:
# 创建微调模型
finetune_net = torchvision.models.resnet18(pretrained=True)

# 替换最后的全连接层 (1000 → 2)
num_classes = 2  # 热狗 vs 非热狗
finetune_net.fc = nn.Linear(finetune_net.fc.in_features, num_classes)

# 初始化新的输出层
nn.init.xavier_uniform_(finetune_net.fc.weight)

print("新的输出层 (2类):")
print(finetune_net.fc)
print(f"\n参数状态:")
print("  - 特征层: 预训练参数 (需微调)")
print("  - 输出层: 随机初始化 (从头训练)")

### 4.4 差异化学习率

In [ ]:
# 设置不同的学习率
# 方法1: 参数分组
feature_params = [p for name, p in finetune_net.named_parameters() 
                 if 'fc' not in name]
output_params = finetune_net.fc.parameters()

optimizer = torch.optim.SGD([
    {'params': feature_params, 'lr': 0.001},  # 特征层: 小lr
    {'params': output_params, 'lr': 0.01}     # 输出层: 大lr (10倍)
], momentum=0.9, weight_decay=5e-4)

print("学习率设置:")
print("  特征层 (预训练): lr = 0.001")
print("  输出层 (新层):   lr = 0.01  (10倍)")
print("\n原理:")
print("  - 预训练参数已经很好,只需微调")
print("  - 新层需要从头学习,需要更大学习率")

### 4.5 训练策略对比

**策略1: 从头训练** (Baseline)
```python
model = ResNet18()  # 随机初始化
# 需要更多数据和时间
# 容易过拟合
```

**策略2: 固定特征提取** (Feature Extraction)
```python
model = ResNet18(pretrained=True)
for param in model.parameters():
    param.requires_grad = False  # 冻结所有层
model.fc = nn.Linear(512, 2)  # 只训练输出层
# 快速,但效果可能不如微调
```

**策略3: 微调** (Fine-tuning) ✅ 推荐
```python
model = ResNet18(pretrained=True)
model.fc = nn.Linear(512, 2)
# 所有层都训练,但特征层用小lr
# 平衡效果和训练成本
```

**策略4: 逐层解冻** (Gradual Unfreezing)
```python
# 先训练输出层
# 再逐步解冻后面的层
# 适合超大模型(BERT, GPT)
```

## 5. 小结

### 图像增广

**核心技术**:
1. **几何变换**: 翻转、裁剪、旋转、缩放
2. **颜色变换**: 亮度、对比度、饱和度、色调
3. **其他**: 噪声、模糊、擦除

**关键原则**:
- ✅ 训练时增广,测试时不增广
- ✅ 根据任务选择合适方法
- ✅ 不要过度增广
- ✅ 使用Compose组合多种方法

**效果**:
- 扩充数据集
- 提高泛化能力
- 防止过拟合
- 提升模型鲁棒性

### 迁移学习

**核心思想**:
```
大数据集(源) → 预训练 → 通用特征
                    ↓
小数据集(目标) → 微调 → 特定任务
```

**微调步骤**:
1. 加载预训练模型
2. 替换输出层
3. 设置差异化学习率
4. 在目标数据集上训练

**学习率策略**:
- 特征层: 小lr (已经训练好)
- 输出层: 大lr (需要从头学)
- 比例: 通常10倍差异

**优势**:
- ✅ 数据需求小
- ✅ 训练时间短
- ✅ 效果通常更好
- ✅ 避免过拟合

### 实践建议

**数据集大小 vs 策略**:
```
超大 (100万+)  → 从头训练
大 (10万+)     → 预训练 + 微调
中 (1万+)      → 微调 (推荐)
小 (几千)      → 微调 + 强增广
超小 (几百)    → 特征提取 or Few-shot
```

**相似度 vs 策略**:
```
相似度高 (如都是自然图像):
  → 微调顶层,冻结底层

相似度中 (如ImageNet→医学图像):
  → 微调所有层,差异化lr

相似度低 (如自然图像→X光):
  → 只用预训练的网络结构
```

### 常用预训练模型

| 模型 | 参数量 | 准确率 | 速度 | 推荐场景 |
|------|--------|--------|------|----------|
| ResNet-18 | 11M | 69.8% | 快 | 快速原型 |
| ResNet-50 | 25M | 76.2% | 中 | 通用任务 |
| ResNet-101 | 45M | 77.4% | 慢 | 高精度需求 |
| EfficientNet-B0 | 5M | 77.1% | 快 | 移动端 |
| EfficientNet-B7 | 66M | 84.4% | 慢 | SOTA精度 |
| ViT-Base | 86M | 81.8% | 中 | Transformer |

### PyTorch加载预训练模型

```python
# 方法1: torchvision
model = torchvision.models.resnet50(pretrained=True)

# 方法2: timm (更多模型)
import timm
model = timm.create_model('efficientnet_b0', pretrained=True)

# 方法3: HuggingFace (Transformers)
from transformers import ViTModel
model = ViTModel.from_pretrained('google/vit-base-patch16-224')
```

## 练习

1. **增广实验**: 对比有无图像增广对CIFAR-10训练的影响。

2. **增广强度**: 尝试不同强度的ColorJitter,观察对训练的影响。

3. **自定义增广**: 实现一个新的增广方法(如随机擦除Cutout)。

4. **微调实验**: 对比冻结特征层vs微调所有层的效果。

5. **学习率**: 实验特征层和输出层不同的学习率比例。

6. **不同模型**: 尝试ResNet-50, EfficientNet等其他预训练模型。

7. **真实任务**: 在自己的小数据集上应用迁移学习。

8. **数据增广库**: 探索albumentations, imgaug等专业增广库。